In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [2]:
PROJECT_ROOT = Path("..")

SEED_FILE = PROJECT_ROOT / "data" / "seed_candidates" / "self_assertion_seeds_v0.csv"

DEV_FILE = PROJECT_ROOT / "data" / "preprocessed" / "development_sentences_clean.csv"

OUTPUT_DIR = PROJECT_ROOT / "data" / "seed_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
seeds = pd.read_csv(SEED_FILE)
dev = pd.read_csv(DEV_FILE)

print(seeds)
print(dev.shape)
print(dev.columns.tolist())

              seed
0   self-assertion
1    assertiveness
2        assertive
3           reject
4           resist
5         advocate
6          propose
7           pursue
8        influence
9           defend
10          demand
11        reaffirm
12          insist
13          assert
14            urge
15          refuse
16          oppose
17          uphold
18           claim
19         declare
(157346, 9)
['corpus', 'iso3', 'session', 'year', 'filename', 'sentence_id', 'sentence_text', 'sentence_text_lower', 'procedural_intro']


In [5]:
TEXT_COL = "sentence_text"
TEXT_LOWER_COL = "sentence_text_lower"
COUNTRY_COL = "iso3"
YEAR_COL = "year"

In [6]:
def seed_mask(series, seed):
    """
    Return True for rows where the exact seed occurs as a whole word/token.
    Case-insensitive because we use sentence_text_lower.
    """
    seed = seed.lower()
    pattern = rf"(?<!\w){re.escape(seed)}(?!\w)"
    return series.fillna("").str.contains(pattern, regex=True)

In [7]:
seed_counts = []

for seed in seeds["seed"]:
    mask = seed_mask(dev[TEXT_LOWER_COL], seed)

    seed_counts.append({
        "seed": seed,
        "n_sentences": int(mask.sum())
    })

seed_counts = pd.DataFrame(seed_counts).sort_values(
    "n_sentences",
    ascending=False
)

seed_counts

,seed,n_sentences
14,urge,694
11,reaffirm,647
10,demand,497
9,defend,433
17,uphold,420
7,pursue,346
5,advocate,304
3,reject,261
18,claim,210
8,influence,194


In [8]:
seed_stats = []

for seed in seeds["seed"]:
    mask = seed_mask(dev[TEXT_LOWER_COL], seed)
    subset = dev.loc[mask]

    n_country_years = (
        subset[[COUNTRY_COL, YEAR_COL]]
        .drop_duplicates()
        .shape[0]
    )

    n_countries = subset[COUNTRY_COL].nunique()

    seed_stats.append({
        "seed": seed,
        "n_sentences": len(subset),
        "n_countries": n_countries,
        "n_country_years": n_country_years
    })

seed_stats = pd.DataFrame(seed_stats).sort_values(
    "n_sentences",
    ascending=False
)

seed_stats

,seed,n_sentences,n_countries,n_country_years
14,urge,694,155,488
11,reaffirm,647,147,455
10,demand,497,146,376
9,defend,433,126,299
17,uphold,420,147,330
7,pursue,346,133,294
5,advocate,304,119,255
3,reject,261,87,183
18,claim,210,91,163
8,influence,194,105,183


In [9]:
seed_categories = {
    "self-assertion": "A",
    "assertiveness": "A",
    "assertive": "A",
    "assert": "A/C",

    "influence": "B",

    "reject": "C",
    "resist": "C",
    "advocate": "C",
    "propose": "C",
    "pursue": "C",
    "defend": "C",
    "demand": "C",
    "reaffirm": "C",
    "insist": "C",
    "urge": "C",
    "refuse": "C",
    "oppose": "C",
    "uphold": "C",
    "claim": "C",
    "declare": "C",
}

seeds["category"] = seeds["seed"].map(seed_categories)

seeds

,seed,category
0,self-assertion,A
1,assertiveness,A
2,assertive,A
3,reject,C
4,resist,C
5,advocate,C
6,propose,C
7,pursue,C
8,influence,B
9,defend,C


In [10]:
seeds.to_csv(
    OUTPUT_DIR / "self_assertion_seed_metadata.csv",
    index=False
)

In [11]:
dev_context = dev.sort_values(
    ["filename", "sentence_id"]
).copy()

dev_context["previous_sentence"] = (
    dev_context.groupby("filename")["sentence_text"].shift(1)
)

dev_context["next_sentence"] = (
    dev_context.groupby("filename")["sentence_text"].shift(-1)
)

dev_context[
    [
        "filename",
        "sentence_id",
        "previous_sentence",
        "sentence_text",
        "next_sentence"
    ]
].head()

,filename,sentence_id,previous_sentence,sentence_text,next_sentence
0,AFG_71_2016.txt,0,NaN,It is my distinct pleasure to attend the seven...,I sincerely congratulate Ambassador Peter Thom...
1,AFG_71_2016.txt,1,It is my distinct pleasure to attend the seven...,I sincerely congratulate Ambassador Peter Thom...,I thank Secretary-General Ban Ki-moon for his ...
2,AFG_71_2016.txt,2,I sincerely congratulate Ambassador Peter Thom...,I thank Secretary-General Ban Ki-moon for his ...,We wish him every success in his future endeav...
3,AFG_71_2016.txt,3,I thank Secretary-General Ban Ki-moon for his ...,We wish him every success in his future endeav...,The twenty-first century has brought with it a...
4,AFG_71_2016.txt,4,We wish him every success in his future endeav...,The twenty-first century has brought with it a...,We have come to realize that dialogue and coop...


In [12]:
N_PER_SEED = 20
RANDOM_STATE = 42

audit_samples = []

for seed in seeds["seed"]:

    mask = seed_mask(
        dev_context["sentence_text_lower"],
        seed
    )

    subset = dev_context.loc[mask].copy()

    if len(subset) == 0:
        print(f"{seed}: 0 occurrences — no contexts available")
        continue

    sample_n = min(N_PER_SEED, len(subset))

    sample = subset.sample(
        n=sample_n,
        random_state=RANDOM_STATE
    ).copy()

    sample["seed"] = seed

    audit_samples.append(sample)

audit_df = pd.concat(
    audit_samples,
    ignore_index=True
)

audit_df.shape

assertiveness: 0 occurrences — no contexts available


(346, 12)

In [13]:
audit_df = audit_df[
    [
        "seed",
        "iso3",
        "year",
        "filename",
        "sentence_id",
        "previous_sentence",
        "sentence_text",
        "next_sentence"
    ]
].copy()

In [14]:
audit_df["construct_relevant"] = ""
audit_df["actor_type"] = ""
audit_df["agency_present"] = ""
audit_df["own_position_or_object"] = ""
audit_df["sense_correct"] = ""
audit_df["boilerplate"] = ""
audit_df["notes"] = ""

In [15]:
audit_df = audit_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [16]:
AUDIT_FILE = OUTPUT_DIR / "self_assertion_seed_manual_audit.csv"

audit_df.to_csv(
    AUDIT_FILE,
    index=False
)

print(AUDIT_FILE)
print(audit_df.shape)

../data/seed_audit/self_assertion_seed_manual_audit.csv
(346, 15)


In [17]:
coverage_map = {
    "urge": "advance_outcome",
    "propose": "advance_outcome",
    "advocate": "advance_outcome",
    "pursue": "advance_outcome",
    "demand": "advance_outcome",

    "assert": "claim_position",
    "claim": "claim_position",
    "declare": "claim_position",

    "defend": "defend_self_relevant",
    "uphold": "defend_self_relevant",

    "reaffirm": "maintain_position",
    "insist": "maintain_position",

    "reject": "resist_constraint",
    "resist": "resist_constraint",
    "refuse": "resist_constraint",
    "oppose": "resist_constraint",

    "influence": "exercise_influence",

    "self-assertion": "construct_label",
    "assertiveness": "construct_label",
    "assertive": "construct_label",
}

coverage_df = pd.DataFrame(
    coverage_map.items(),
    columns=["seed", "facet"]
)

coverage_df

,seed,facet
0,urge,advance_outcome
1,propose,advance_outcome
2,advocate,advance_outcome
3,pursue,advance_outcome
4,demand,advance_outcome
5,assert,claim_position
6,claim,claim_position
7,declare,claim_position
8,defend,defend_self_relevant
9,uphold,defend_self_relevant


In [20]:
audit_summary = pd.read_csv(
    OUTPUT_DIR / "Self Assertion Audit Summary.csv"
)

coverage_df = coverage_df.merge(
    audit_summary,
    on="seed",
    how="left"
)

coverage_df

,seed,facet,n,relevant_yes,relevant_no,sense_correct,focal_actor,boilerplate_yes,precision
0,urge,advance_outcome,20.0,20.0,0.0,20.0,20.0,0.0,1.00
1,propose,advance_outcome,20.0,17.0,3.0,20.0,17.0,0.0,0.85
2,advocate,advance_outcome,20.0,14.0,6.0,19.0,15.0,1.0,0.70
3,pursue,advance_outcome,20.0,5.0,15.0,20.0,5.0,0.0,0.25
4,demand,advance_outcome,20.0,7.0,13.0,14.0,7.0,0.0,0.35
5,assert,claim_position,20.0,7.0,13.0,20.0,7.0,0.0,0.35
6,claim,claim_position,20.0,1.0,19.0,10.0,1.0,0.0,0.05
7,declare,claim_position,20.0,11.0,9.0,19.0,11.0,0.0,0.55
8,defend,defend_self_relevant,20.0,5.0,15.0,20.0,5.0,0.0,0.25
9,uphold,defend_self_relevant,20.0,6.0,14.0,19.0,6.0,0.0,0.30


In [21]:
facet_summary = (
    coverage_df[
        coverage_df["facet"] != "construct_label"
    ]
    .groupby("facet")
    .agg(
        n_seeds=("seed", "count"),
        mean_precision=("precision", "mean"),
        max_precision=("precision", "max")
    )
    .reset_index()
)

facet_summary

,facet,n_seeds,mean_precision,max_precision
0,advance_outcome,5,0.630000,1.00
1,claim_position,3,0.316667,0.55
2,defend_self_relevant,2,0.275000,0.30
3,exercise_influence,1,0.100000,0.10
4,maintain_position,2,0.625000,0.65
5,resist_constraint,4,0.462500,0.75


In [22]:
all_facets = [
    "advance_outcome",
    "claim_position",
    "defend_self_relevant",
    "maintain_position",
    "resist_constraint",
    "commit_intended_action",
    "exercise_influence",
]

facet_summary = (
    pd.DataFrame({"facet": all_facets})
    .merge(
        facet_summary,
        on="facet",
        how="left"
    )
)

facet_summary["n_seeds"] = facet_summary["n_seeds"].fillna(0).astype(int)

facet_summary

,facet,n_seeds,mean_precision,max_precision
0,advance_outcome,5,0.630000,1.00
1,claim_position,3,0.316667,0.55
2,defend_self_relevant,2,0.275000,0.30
3,maintain_position,2,0.625000,0.65
4,resist_constraint,4,0.462500,0.75
5,commit_intended_action,0,NaN,NaN
6,exercise_influence,1,0.100000,0.10


In [23]:
def classify_facet(row):
    if row["n_seeds"] == 0:
        return "missing"

    if row["max_precision"] >= 0.70:
        return "strong"

    if row["max_precision"] >= 0.40:
        return "moderate"

    return "weak"


facet_summary["coverage_status"] = facet_summary.apply(
    classify_facet,
    axis=1
)

facet_summary

,facet,n_seeds,mean_precision,max_precision,coverage_status
0,advance_outcome,5,0.630000,1.00,strong
1,claim_position,3,0.316667,0.55,moderate
2,defend_self_relevant,2,0.275000,0.30,weak
3,maintain_position,2,0.625000,0.65,moderate
4,resist_constraint,4,0.462500,0.75,strong
5,commit_intended_action,0,NaN,NaN,missing
6,exercise_influence,1,0.100000,0.10,weak


In [24]:
coverage_df[
    coverage_df["facet"] != "construct_label"
][
    ["facet", "seed", "precision", "focal_actor", "sense_correct", "boilerplate_yes"]
].sort_values(
    ["facet", "precision"],
    ascending=[True, False]
)

,facet,seed,precision,focal_actor,sense_correct,boilerplate_yes
0,advance_outcome,urge,1.00,20.0,20.0,0.0
1,advance_outcome,propose,0.85,17.0,20.0,0.0
2,advance_outcome,advocate,0.70,15.0,19.0,1.0
4,advance_outcome,demand,0.35,7.0,14.0,0.0
3,advance_outcome,pursue,0.25,5.0,20.0,0.0
7,claim_position,declare,0.55,11.0,19.0,0.0
5,claim_position,assert,0.35,7.0,20.0,0.0
6,claim_position,claim,0.05,1.0,10.0,0.0
9,defend_self_relevant,uphold,0.30,6.0,19.0,0.0
8,defend_self_relevant,defend,0.25,5.0,20.0,0.0


In [25]:
facet_summary.to_csv(
    OUTPUT_DIR / "Self Assertion Facet Coverage Summary.csv",
    index=False
)

In [26]:
SELF_ASSERTION_FACETS = [
    "advance_outcome",
    "claim_position",
    "defend_self_relevant",
    "maintain_position",
    "resist_constraint",
    "commit_intended_action",
    "exercise_capability_role",
    "influence_outcome",
    "other"
]

In [27]:
N_COVERAGE_AUDIT = 250
RANDOM_STATE_COVERAGE = 123

coverage_sample = dev_context.sample(
    n=N_COVERAGE_AUDIT,
    random_state=RANDOM_STATE_COVERAGE
).copy()

In [28]:
coverage_sample = coverage_sample[
    [
        "iso3",
        "year",
        "filename",
        "sentence_id",
        "previous_sentence",
        "sentence_text",
        "next_sentence"
    ]
].copy()

In [29]:
coverage_sample = coverage_sample.sample(
    frac=1,
    random_state=456
).reset_index(drop=True)

In [30]:
coverage_sample["is_self_assertion"] = ""
coverage_sample["facet"] = ""
coverage_sample["key_expression"] = ""
coverage_sample["covered_by_current_seed_family"] = ""
coverage_sample["coverage_notes"] = ""

In [31]:
COVERAGE_FILE = (
    OUTPUT_DIR /
    "Self Assertion Independent Coverage Audit.csv"
)

coverage_sample.to_csv(
    COVERAGE_FILE,
    index=False
)

print(COVERAGE_FILE)
print(coverage_sample.shape)

../data/seed_audit/Self Assertion Independent Coverage Audit.csv
(250, 12)


In [32]:
coverage_coded = pd.read_csv(
    OUTPUT_DIR / "Self Assertion Independent Coverage Audit_CODED.csv"
)

coverage_coded.head()

,iso3,year,filename,sentence_id,previous_sentence,sentence_text,next_sentence,is_self_assertion,facet,key_expression,covered_by_current_seed_family,coverage_notes
0,YEM,2020,YEM_75_2020.txt,31,What is even worse is that this militia took t...,What you see today is a ruthless escalation by...,It attacks residential areas with ballistic mi...,no,NaN,NaN,NaN,No focal-national agentive self-positioning in...
1,GRC,2022,GRC_77_2022.txt,51,It is also the only NATO country not implement...,"When it comes to Greece, Tiirkiye’s leadership...",Their language is increasingly bellicose.,no,NaN,NaN,NaN,No focal-national agentive self-positioning in...
2,SGP,2020,SGP_75_2020.txt,71,We have a responsibility to help our people to...,We also have to help our businesses leverage d...,"At the international level, governments, busin...",yes,commit_intended_action,have to help our businesses,no,Focal actor states an intended responsibility/...
3,MCO,2023,MCO_78_2023.txt,49,What major countries have achieved in space sh...,While we look for the presence of water on the...,The young people of our nations place before u...,no,NaN,NaN,NaN,No focal-national agentive self-positioning in...
4,DZA,2023,DZA_78_2023.txt,76,my country issued its fourth national report i...,My country is firmly convinced that respect fo...,That is why we work to strengthen human rights...,no,NaN,NaN,NaN,No focal-national agentive self-positioning in...


In [33]:
coverage_gaps = coverage_coded[
    (coverage_coded["is_self_assertion"] == "yes") &
    (coverage_coded["covered_by_current_seed_family"] == "no")
].copy()

coverage_gaps[
    [
        "facet",
        "key_expression",
        "sentence_text"
    ]
]

,facet,key_expression,sentence_text
2,commit_intended_action,have to help our businesses,We also have to help our businesses leverage d...
7,commit_intended_action,readiness to offer help,"Nevertheless, in all our actions aimed at solv..."
8,exercise_capability_role,making its own way,Belarus remains on pace in its successful deve...
38,exercise_capability_role,issued two sovereign green bonds,We have since issued two sovereign green bonds...
39,exercise_capability_role,actively participated in the negotiations,It is for that reason that we actively partici...
47,commit_intended_action,committed to moving towards,The message that we want to convey is clear: d...
63,commit_intended_action,committed to continuing our work,The pandemic has made this work only more rele...
66,commit_intended_action,personally committed to that fight,That first commitment on the part of our Organ...
67,commit_intended_action,commitment to multilateralism,COMMITMENT / WHAT BELGIUM IS DOING
77,commit_intended_action,will continue to support,Spain will continue to support all efforts aim...


In [34]:
gap_expression_counts = (
    coverage_gaps
    .groupby(["facet", "key_expression"])
    .size()
    .reset_index(name="n")
    .sort_values(
        ["facet", "n"],
        ascending=[True, False]
    )
)

gap_expression_counts

,facet,key_expression,n
17,commit_intended_action,will continue to support,2
0,commit_intended_action,I will do my utmost to achieve that,1
1,commit_intended_action,affirm... commitment... to the roadmap,1
2,commit_intended_action,commitment to multilateralism,1
3,commit_intended_action,committed to continuing our work,1
4,commit_intended_action,committed to moving towards,1
5,commit_intended_action,have to help our businesses,1
6,commit_intended_action,intent on reviving,1
7,commit_intended_action,personally committed to that fight,1
8,commit_intended_action,readiness to offer help,1


In [35]:
coverage_gaps[
    coverage_gaps["facet"] == "commit_intended_action"
][
    ["key_expression", "sentence_text"]
]

,key_expression,sentence_text
2,have to help our businesses,We also have to help our businesses leverage d...
7,readiness to offer help,"Nevertheless, in all our actions aimed at solv..."
47,committed to moving towards,The message that we want to convey is clear: d...
63,committed to continuing our work,The pandemic has made this work only more rele...
66,personally committed to that fight,That first commitment on the part of our Organ...
67,commitment to multilateralism,COMMITMENT / WHAT BELGIUM IS DOING
77,will continue to support,Spain will continue to support all efforts aim...
87,stands ready to cooperate,I would also like to assure him that my delega...
96,we will act to fulfil,We will act to fulfil the aspirations of our p...
108,we will invest,"With the ecological transformation plan, we wi..."


In [36]:
coverage_gaps[
    coverage_gaps["facet"] == "exercise_capability_role"
][
    ["key_expression", "sentence_text"]
]

,key_expression,sentence_text
8,making its own way,Belarus remains on pace in its successful deve...
38,issued two sovereign green bonds,We have since issued two sovereign green bonds...
39,actively participated in the negotiations,It is for that reason that we actively partici...
94,does not shrink from global challenge,This is something that still guides us.
104,our country plays a special role,I highlighted the Secretary-General’s reforms ...
132,fully embrace our national sovereignty,In acknowledging the end to an imbalanced and ...
144,focused on mediation,We have also focused on mediation to settle di...
175,engaged in crisis prevention,Its engagement is focused on humanitarian aid;...
205,complete... transparency... submit the third NDC,We were the first to complete the initial cycl...


In [37]:
GAP_SEED_FILE = (
    PROJECT_ROOT
    / "data"
    / "seed_candidates"
    / "self_assertion_gap_candidates.csv"
)

gap_seeds = pd.read_csv(GAP_SEED_FILE)

gap_seeds

,seed,facet
0,commit,commit_intended_action
1,intend,commit_intended_action
2,ready,commit_intended_action
3,continue,commit_intended_action
4,pledge,commit_intended_action
5,exercise,exercise_capability_role
6,participate,exercise_capability_role
7,engage,exercise_capability_role
8,mediate,exercise_capability_role
9,lead,exercise_capability_role


In [38]:
gap_stats = []

for seed in gap_seeds["seed"]:
    mask = seed_mask(
        dev_context["sentence_text_lower"],
        seed
    )

    subset = dev_context.loc[mask]

    gap_stats.append({
        "seed": seed,
        "n_sentences": len(subset),
        "n_countries": subset["iso3"].nunique(),
        "n_country_years": (
            subset[["iso3", "year"]]
            .drop_duplicates()
            .shape[0]
        )
    })

gap_stats = pd.DataFrame(gap_stats).sort_values(
    "n_sentences",
    ascending=False
)

gap_stats

,seed,n_sentences,n_countries,n_country_years
3,continue,3947,178,1444
9,lead,911,171,676
2,ready,672,152,491
6,participate,423,145,360
7,engage,417,145,329
5,exercise,309,118,255
0,commit,290,127,249
4,pledge,213,99,183
1,intend,134,70,105
8,mediate,9,7,7


In [39]:
N_PER_GAP_SEED = 20
RANDOM_STATE_GAP = 789

gap_audit_samples = []

for seed in gap_seeds["seed"]:
    mask = seed_mask(
        dev_context["sentence_text_lower"],
        seed
    )

    subset = dev_context.loc[mask].copy()

    if len(subset) == 0:
        continue

    sample_n = min(N_PER_GAP_SEED, len(subset))

    sample = subset.sample(
        n=sample_n,
        random_state=RANDOM_STATE_GAP
    ).copy()

    sample["seed"] = seed

    # attach intended facet from candidate file
    facet = gap_seeds.loc[
        gap_seeds["seed"] == seed,
        "facet"
    ].iloc[0]

    sample["target_facet"] = facet

    gap_audit_samples.append(sample)

gap_audit_df = pd.concat(
    gap_audit_samples,
    ignore_index=True
)

gap_audit_df.shape

(189, 13)

In [40]:
gap_audit_df = gap_audit_df[
    [
        "seed",
        "target_facet",
        "iso3",
        "year",
        "filename",
        "sentence_id",
        "previous_sentence",
        "sentence_text",
        "next_sentence"
    ]
].copy()

In [41]:
gap_audit_df["construct_relevant"] = ""
gap_audit_df["facet_relevant"] = ""
gap_audit_df["actor_type"] = ""
gap_audit_df["agency_present"] = ""
gap_audit_df["sense_correct"] = ""
gap_audit_df["boilerplate"] = ""
gap_audit_df["notes"] = ""

In [42]:
gap_audit_df = gap_audit_df.sample(
    frac=1,
    random_state=789
).reset_index(drop=True)

In [43]:
GAP_AUDIT_FILE = (
    OUTPUT_DIR /
    "Self Assertion Gap Candidate Audit.csv"
)

gap_audit_df.to_csv(
    GAP_AUDIT_FILE,
    index=False
)

print(GAP_AUDIT_FILE)
print(gap_audit_df.shape)

../data/seed_audit/Self Assertion Gap Candidate Audit.csv
(189, 16)


In [44]:
original_summary = pd.read_csv(
    OUTPUT_DIR / "Self Assertion Audit Summary.csv"
)

gap_summary = pd.read_csv(
    OUTPUT_DIR / "Self Assertion Gap Candidate Audit_SUMMARY.csv"
)

In [45]:
original_summary.head()
gap_summary.head()

,seed,target_facet,n,construct_yes,facet_yes,sense_correct,focal_actor,boilerplate_yes,construct_precision,facet_precision
0,intend,commit_intended_action,20,15,15,20,15,0,0.75,0.75
1,ready,commit_intended_action,20,14,14,20,14,1,0.70,0.70
2,continue,commit_intended_action,20,14,13,16,14,0,0.70,0.65
3,commit,commit_intended_action,20,7,5,16,7,0,0.35,0.25
4,pledge,commit_intended_action,20,4,3,20,4,4,0.20,0.15


In [46]:
original_decisions = original_summary[
    ["seed", "precision"]
].copy()

original_decisions["source"] = "original"
original_decisions["facet_precision"] = original_decisions["precision"]
original_decisions = original_decisions.drop(columns="precision")

In [47]:
gap_decisions = gap_summary[
    ["seed", "target_facet", "facet_precision"]
].copy()

gap_decisions = gap_decisions.rename(
    columns={"target_facet": "facet"}
)

gap_decisions["source"] = "gap_added"

In [48]:
original_decisions = original_decisions.merge(
    coverage_df[["seed", "facet"]].drop_duplicates(),
    on="seed",
    how="left"
)

In [49]:
decision_df = pd.concat(
    [
        original_decisions[["seed", "facet", "facet_precision", "source"]],
        gap_decisions[["seed", "facet", "facet_precision", "source"]]
    ],
    ignore_index=True
)

decision_df

,seed,facet,facet_precision,source
0,urge,advance_outcome,1.000000,original
1,propose,advance_outcome,0.850000,original
2,reject,resist_constraint,0.750000,original
3,advocate,advance_outcome,0.700000,original
4,reaffirm,maintain_position,0.650000,original
5,insist,maintain_position,0.600000,original
6,declare,claim_position,0.550000,original
7,resist,resist_constraint,0.550000,original
8,assert,claim_position,0.350000,original
9,demand,advance_outcome,0.350000,original


In [50]:
def seed_decision(p):
    if pd.isna(p):
        return "theory_anchor"
    elif p >= 0.70:
        return "keep"
    elif p >= 0.40:
        return "borderline"
    else:
        return "exclude"

decision_df["preliminary_decision"] = (
    decision_df["facet_precision"].apply(seed_decision)
)

decision_df.sort_values(
    ["facet", "facet_precision"],
    ascending=[True, False]
)

,seed,facet,facet_precision,source,preliminary_decision
0,urge,advance_outcome,1.000000,original,keep
1,propose,advance_outcome,0.850000,original,keep
3,advocate,advance_outcome,0.700000,original,keep
9,demand,advance_outcome,0.350000,original,exclude
13,pursue,advance_outcome,0.250000,original,exclude
6,declare,claim_position,0.550000,original,borderline
8,assert,claim_position,0.350000,original,exclude
17,claim,claim_position,0.050000,original,exclude
19,intend,commit_intended_action,0.750000,gap_added,keep
20,ready,commit_intended_action,0.700000,gap_added,keep


In [51]:
final_decisions = {
    # theory anchors
    "self-assertion": "theory_anchor",
    "assertiveness": "theory_anchor",
    "assertive": "theory_anchor",
    "assert": "theory_anchor",

    # retained empirical anchors
    "urge": "keep",
    "propose": "keep",
    "advocate": "keep",
    "declare": "keep",
    "intend": "keep",
    "ready": "keep",
    "reaffirm": "keep",
    "insist": "keep",
    "reject": "keep",
    "resist": "keep",

    # hold out
    "mediate": "hold_out",

    # exclude
    "demand": "exclude",
    "pursue": "exclude",
    "claim": "exclude",
    "defend": "exclude",
    "uphold": "exclude",
    "influence": "exclude",
    "continue": "exclude",
    "commit": "exclude",
    "pledge": "exclude",
    "engage": "exclude",
    "participate": "exclude",
    "exercise": "exclude",
    "lead": "exclude",
    "oppose": "exclude",
    "refuse": "exclude",
}

decision_df["final_decision"] = (
    decision_df["seed"].map(final_decisions)
)

decision_df.sort_values(
    ["final_decision", "facet", "seed"]
)

,seed,facet,facet_precision,source,preliminary_decision,final_decision
9,demand,advance_outcome,0.350000,original,exclude,exclude
13,pursue,advance_outcome,0.250000,original,exclude,exclude
17,claim,claim_position,0.050000,original,exclude,exclude
22,commit,commit_intended_action,0.250000,gap_added,exclude,exclude
21,continue,commit_intended_action,0.650000,gap_added,borderline,exclude
23,pledge,commit_intended_action,0.150000,gap_added,exclude,exclude
12,defend,defend_self_relevant,0.250000,original,exclude,exclude
11,uphold,defend_self_relevant,0.300000,original,exclude,exclude
25,engage,exercise_capability_role,0.300000,gap_added,exclude,exclude
27,exercise,exercise_capability_role,0.150000,gap_added,exclude,exclude


In [52]:
seed_set_v1 = decision_df[
    decision_df["final_decision"].isin(
        ["keep", "theory_anchor"]
    )
].copy()

seed_set_v1[
    ["seed", "facet", "source", "final_decision"]
]

,seed,facet,source,final_decision
0,urge,advance_outcome,original,keep
1,propose,advance_outcome,original,keep
2,reject,resist_constraint,original,keep
3,advocate,advance_outcome,original,keep
4,reaffirm,maintain_position,original,keep
5,insist,maintain_position,original,keep
6,declare,claim_position,original,keep
7,resist,resist_constraint,original,keep
8,assert,claim_position,original,theory_anchor
15,assertive,construct_label,original,theory_anchor


In [53]:
seed_set_v1.to_csv(
    PROJECT_ROOT
    / "data"
    / "seed_candidates"
    / "self_assertion_seed_set_v1.csv",
    index=False
)

In [54]:
final_decisions["defend"] = "theory_anchor"
final_decisions["exercise"] = "theory_anchor"
final_decisions["assertiveness"] = "theory_anchor"

In [55]:
decision_df["final_decision"] = (
    decision_df["seed"].map(final_decisions)
)

seed_set_v1 = decision_df[
    decision_df["final_decision"].isin(
        ["keep", "theory_anchor"]
    )
].copy()

seed_set_v1[
    ["seed", "facet", "source", "final_decision"]
]

,seed,facet,source,final_decision
0,urge,advance_outcome,original,keep
1,propose,advance_outcome,original,keep
2,reject,resist_constraint,original,keep
3,advocate,advance_outcome,original,keep
4,reaffirm,maintain_position,original,keep
5,insist,maintain_position,original,keep
6,declare,claim_position,original,keep
7,resist,resist_constraint,original,keep
8,assert,claim_position,original,theory_anchor
12,defend,defend_self_relevant,original,theory_anchor


In [56]:
final_decisions = {
    # keep
    "urge": "keep",
    "propose": "keep",
    "advocate": "keep",
    "declare": "keep",
    "reaffirm": "keep",
    "insist": "keep",
    "reject": "keep",
    "resist": "keep",
    "intend": "keep",
    "ready": "keep",

    # theory anchors
    "self-assertion": "theory_anchor",
    "assert": "theory_anchor",
    "defend": "theory_anchor",
    "exercise": "theory_anchor",

    # exclude
    "assertive": "exclude",
    "assertiveness": "exclude",
    "demand": "exclude",
    "pursue": "exclude",
    "claim": "exclude",
    "uphold": "exclude",
    "influence": "exclude",
    "continue": "exclude",
    "commit": "exclude",
    "pledge": "exclude",
    "engage": "exclude",
    "participate": "exclude",
    "lead": "exclude",
    "oppose": "exclude",
    "refuse": "exclude",
    "mediate": "exclude",
}

In [57]:
decision_df["final_decision"] = (
    decision_df["seed"].map(final_decisions)
)

decision_df.sort_values(
    ["final_decision", "facet", "seed"]
)

,seed,facet,facet_precision,source,preliminary_decision,final_decision
9,demand,advance_outcome,0.350000,original,exclude,exclude
13,pursue,advance_outcome,0.250000,original,exclude,exclude
17,claim,claim_position,0.050000,original,exclude,exclude
22,commit,commit_intended_action,0.250000,gap_added,exclude,exclude
21,continue,commit_intended_action,0.650000,gap_added,borderline,exclude
23,pledge,commit_intended_action,0.150000,gap_added,exclude,exclude
15,assertive,construct_label,0.200000,original,exclude,exclude
11,uphold,defend_self_relevant,0.300000,original,exclude,exclude
25,engage,exercise_capability_role,0.300000,gap_added,exclude,exclude
28,lead,exercise_capability_role,0.150000,gap_added,exclude,exclude


In [58]:
seed_set_v1 = decision_df[
    decision_df["final_decision"].isin(
        ["keep", "theory_anchor"]
    )
].copy()

seed_set_v1[
    ["seed", "facet", "source", "final_decision"]
]

,seed,facet,source,final_decision
0,urge,advance_outcome,original,keep
1,propose,advance_outcome,original,keep
2,reject,resist_constraint,original,keep
3,advocate,advance_outcome,original,keep
4,reaffirm,maintain_position,original,keep
5,insist,maintain_position,original,keep
6,declare,claim_position,original,keep
7,resist,resist_constraint,original,keep
8,assert,claim_position,original,theory_anchor
12,defend,defend_self_relevant,original,theory_anchor


In [59]:
SEED_V1_FILE = (
    PROJECT_ROOT
    / "data"
    / "seed_candidates"
    / "self_assertion_seed_set_v1.csv"
)

seed_set_v1[
    ["seed", "facet", "source", "final_decision"]
].to_csv(
    SEED_V1_FILE,
    index=False
)

print(SEED_V1_FILE)

../data/seed_candidates/self_assertion_seed_set_v1.csv


In [60]:
DECISION_FILE = (
    OUTPUT_DIR
    / "Self Assertion Seed Final Decisions.csv"
)

decision_df.to_csv(
    DECISION_FILE,
    index=False
)

print(DECISION_FILE)

../data/seed_audit/Self Assertion Seed Final Decisions.csv


In [61]:
final_facet_coverage = (
    seed_set_v1
    .groupby("facet")["seed"]
    .apply(list)
    .reset_index(name="retained_seeds")
)

final_facet_coverage

,facet,retained_seeds
0,advance_outcome,"[urge, propose, advocate]"
1,claim_position,"[declare, assert]"
2,commit_intended_action,"[intend, ready]"
3,construct_label,[self-assertion]
4,defend_self_relevant,[defend]
5,exercise_capability_role,[exercise]
6,maintain_position,"[reaffirm, insist]"
7,resist_constraint,"[reject, resist]"
